<img src='https://data.inpe.br/wp-content/uploads/2025/06/big-1.svg' align='right' width='100px'/>

# <span style='color:#336699;'>Leitura Parcial de Imagens GOES-19 utilizando xarray</span>
<hr style='border:2px solid #0077b9;'>

<div style='text-align: center;font-size: 90%;'>
    Douglas Uba<br/><br/>
    Programa BIG - Base de Informações Georreferenciadas do INPE<br/>
    DISSM -  Divisão de Satélites e Sensores Meteorológicos.
    <br/>
    CGCT - Coordenação-Geral de Ciências da Terra.
    <br/>
    INPE - Instituto Nacional de Pesquisas Espaciais, Brasil.
    <br/>
    Contato: <a href='mailto:douglas.uba@inpe.br'>douglas.uba@inpe.br</a>
    <br/><br/>
    Última Atualização: 15 de abril de 2026.
</div>

<br/>

<div style='text-align: justify;  margin-left: 25%; margin-right: 25%;'>
<b>Resumo</b> Este notebook demonstra a descoberta e leitura de imagens do satélite GOES-19 a partir do Catálogo Integrado do INPE utilizando o padrão STAC. O objetivo é apresentar o acesso aos dados em Python com a biblioteca xarray, realizando leituras espaciais parciais definidas por uma região de interesse (extent), permitindo acessar apenas o subconjunto da imagem necessário, sem transferir o arquivo completo. Importante: a  metodologia de leitura apresentada pode ser aplicada também para o satélite GOES-16. 
</div>

## 👩🏽‍💻 STAC Client API
<hr style='border:1px solid #0077b9;'>

Para execução dos exemplos deste Jupyter Notebook, será instalado o pacote [pystac-client](https://pystac-client.readthedocs.io/en/latest/).

In [ ]:
# Não necessário no ambiente do BDC-Lab
#!pip install pystac-client

Para acessar as funcionalidades, importa-se o pacote `pystac_client`:

In [ ]:
import pystac_client
pystac_client.__version__

Em seguida, realiza-se a conexão com o serviço STAC do INPE:

In [ ]:
service = pystac_client.Client.open(
    'https://data.inpe.br/bdc/stac/v1/'
)
service

## 🔍 Recuperando imagens do dia 08/04/2026, 13h UTC
<hr style='border:1px solid #0077b9;'>

Utilizando o serviço STAC, partir do método `search`, faremos a recuperação de `Items` da coleção `GOES19-L2-CMI-1`. Vamos utilizar o parâmetro `datetime` para **selecionar a data e hora de interesse**: `2026-04-08 13:00 UTC`.

In [ ]:
item_search = service.search(
    collections=['GOES19-L2-CMI-1'],
    datetime='2026-04-08T13:10:00',
)

Na sequência, construímos uma lista com o item que foi recuperado, assim podemos associá-lo à variável `item`:

In [ ]:
item = list(item_search.items())[0]

Para o `Item`, temos as imagens (`Asset`) de interesse (`B01`, `B02`, `B03`, ..., `B16`).

In [ ]:
item

As imagens do GOES-R são fornecidas no formato [**Network Common Data Form (NetCDF)**](https://www.unidata.ucar.edu/software/netcdf/), amplamente utilizado para armazenar dados científicos multidimensionais, como variáveis climáticas e ambientais. Neste caso, utilizaremos a biblioteca `xarray` para ler as informações das imagens.

## 🖥️ Leitura de imagens com xarray
<hr style='border:1px solid #0077b9;'>

Utilizaremos a biblioteca `xarray` para ler os dados dos canais (`Assets`) e visualizar a imagem correspondente. Para isto, criamos o método `open_goes_url`, que recebe como parâmetro de entrada a `url` onde o arquivo (*i.e.* `Asset`) está armazenado. Perceba que estamos carregando o conteúdo total do arquivo da banda `B13`.

In [ ]:
import fsspec
import xarray as xr

def open_goes_image(url):
    f = fsspec.open(url, 'rb')
    ds = xr.open_dataset(
        f.open(),
        engine='h5netcdf',
        chunks='auto'
    )
    return ds.load()

image = open_goes_image(item.assets['B13'].href)
image

Visualização da imagem obtida (*i.e.* setor *full-disk*) a partir da variável `CMI`.

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(image.CMI, cmap='Greys', vmin=190.0, vmax=320.0)
plt.show()

Criamos então a função `geo2goes` que é capaz de converter um `extent` em coordenadas geográficas (latitude e longitude) para o sistema de coordenadas do GOES.

In [ ]:
from pyproj import Proj

GOES_PROJ4 = Proj(
    '+proj=geos +h=35786023.0 \
    +a=6378137.0 +b=6356752.31414 \
    +f=0.00335281068119356027 +lat_0=0.0 \
    +lon_0=-75.0 +sweep=x +no_defs'
)
def geo2goes(extent):
    llx, lly, urx, ury = extent
    x0, y0 = GOES_PROJ4(llx, lly)
    x1, y1 = GOES_PROJ4(urx, ury)
    H = 35786023.0
    x0 /= H; x1 /= H; y0 /= H; y1 /= H
    xmin, xmax = sorted([x0, x1])
    ymin, ymax = sorted([y0, y1])
    return xmin, ymin, xmax, ymax

Agora podemos otimizar a função `open_goes_image`. Além da `url` é possível também informar um `extent`. Ou seja, informamos uma região geográfica específica da qual queremos obter os pixels imageados.

In [ ]:
import fsspec
import xarray as xr

def open_goes_image(url, extent=None):
    f = fsspec.open(url, 'rb')
    ds = xr.open_dataset(
        f.open(),
        engine='h5netcdf',
        chunks='auto'
    )
    if extent is None:
        return ds.load()
    # else, convert extent to GOES coordinate system
    xmin, ymin, xmax, ymax = geo2goes(extent)
    # and clip
    subset = ds.sel(
        x=slice(xmin, xmax),
        y=slice(ymax, ymin)
    )
    return subset.load()

De modo a possibilitar a leitura de uma área específica, vamos definir uma região geográfica (`LAT_LONG_WGS84_SOUTH_AMERICA_EXTENT`) que abrange a América do Sul.

In [ ]:
# Define the Area of Interest (llx, lly, urx, ury)
LAT_LONG_WGS84_SOUTH_AMERICA_EXTENT = [-88.02, -46.50, -26.22, 12.54]

Assim, utilizamos a função `open_goes_image` para realizar a leitura somente dessa região geográfica.

In [ ]:
image = open_goes_image(
    item.assets['B13'].href,
    extent=LAT_LONG_WGS84_SOUTH_AMERICA_EXTENT
)
image

Visualização da imagem obtida (*i.e.* região `LAT_LONG_WGS84_SOUTH_AMERICA_EXTENT`) a partir da variável `CMI`.

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(image.CMI, cmap='Greys', vmin=190.0, vmax=320.0)
plt.show()

**Nota:** podemos utilizar a função `open_goes_image` para obter a imagem de qualquer canal espectral disponível. Por exemplo, a banda `B09` (vapor d'água).

In [ ]:
image = open_goes_image(
    item.assets['B09'].href,
    extent=LAT_LONG_WGS84_SOUTH_AMERICA_EXTENT
)
plt.imshow(image.CMI, cmap='Greys', vmin=190.0, vmax=270.0)
plt.show()

Ou então do canal visível, banda `B01`.

In [ ]:
image = open_goes_image(
    item.assets['B01'].href,
    extent=LAT_LONG_WGS84_SOUTH_AMERICA_EXTENT
)
plt.imshow(image.CMI, cmap='gray', vmin=0.0, vmax=1.0)
plt.show()

Podemos também restringir ainda mais a área de interesse, agora para o estado de Minas Gerais, Brasil.

In [ ]:
# Define MG State Area (llx, lly, urx, ury)
LAT_LONG_WGS84_MG_EXTENT = [-51.50, -23.0, -39.85, -14.15]

In [ ]:
image = open_goes_image(
    item.assets['B13'].href,
    extent=LAT_LONG_WGS84_MG_EXTENT
)
image

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(image.CMI, cmap='Greys', vmin=190.0, vmax=320.0)
plt.show()

Por último, fazemos a leitura da banda `B02`, canal com maior resolução espacial do sensor ABI/GOES-19 (*i.e.* 500 metros).

In [ ]:
image = open_goes_image(
    item.assets['B02'].href,
    extent=LAT_LONG_WGS84_MG_EXTENT
)
image

In [ ]:
import matplotlib.pyplot as plt
plt.imshow(image.CMI, cmap='gray', vmin=0.0, vmax=1.0)
plt.show()

Finalizando, vamos obter novamente a imagem para a América do Sul e visualizar utilizando o `Cartopy`, na projeção geoestacionária do GOES.

In [ ]:
import cartopy
import cartopy.crs as ccrs
from cartopy.feature import NaturalEarthFeature

image = open_goes_image(
    item.assets['B13'].href,
    extent=LAT_LONG_WGS84_SOUTH_AMERICA_EXTENT
)

# Projection parameters
proj = image.goes_imager_projection
sat_height = proj.perspective_point_height
lon_0 = proj.longitude_of_projection_origin
sweep = proj.sweep_angle_axis

# Cartopy GOES projection
goes_crs = ccrs.Geostationary(
    central_longitude=lon_0,
    satellite_height=sat_height,
    sweep_axis=sweep
)

# Extent (meters)
x = image.x.values * sat_height
y = image.y.values * sat_height
extent = [x.min(), x.max(), y.min(), y.max()]

# Create figure
fig = plt.figure(figsize=(6,6))
ax = plt.axes(projection=goes_crs)

# First frame
img = ax.imshow(
    image.CMI,
    origin='upper',
    extent=extent,
    transform=goes_crs,
    cmap='Greys',
    vmin=190,
    vmax=320,
    animated=True
)

# Add references
ax.add_feature(cartopy.feature.BORDERS, edgecolor='white', linewidth=1.0)
ax.coastlines(color='white', linewidth=1.0)
states = NaturalEarthFeature(
    category='cultural',
    scale='50m',
    facecolor='none',
    name='admin_1_states_provinces_lines'
)
ax.add_feature(states, edgecolor='gray')

# Add legend
plt.colorbar(img, ax=ax, shrink=0.7)

In [ ]:
import cartopy
import cartopy.crs as ccrs
from cartopy.feature import NaturalEarthFeature

image = open_goes_image(
    item.assets['B02'].href,
    extent=LAT_LONG_WGS84_MG_EXTENT
)

# Projection parameters
proj = image.goes_imager_projection
sat_height = proj.perspective_point_height
lon_0 = proj.longitude_of_projection_origin
sweep = proj.sweep_angle_axis

# Cartopy GOES projection
goes_crs = ccrs.Geostationary(
    central_longitude=lon_0,
    satellite_height=sat_height,
    sweep_axis=sweep
)

# Extent (meters)
x = image.x.values * sat_height
y = image.y.values * sat_height
extent = [x.min(), x.max(), y.min(), y.max()]

# Create figure
fig = plt.figure(figsize=(6,6))
ax = plt.axes(projection=goes_crs)

# First frame
img = ax.imshow(
    image.CMI,
    origin='upper',
    extent=extent,
    transform=goes_crs,
    cmap='gray',
    vmin=0.0,
    vmax=1.0,
    animated=True
)

# Add references
ax.add_feature(cartopy.feature.BORDERS, edgecolor='white', linewidth=1.0)
ax.coastlines(color='white', linewidth=1.0)
states = NaturalEarthFeature(
    category='cultural',
    scale='50m',
    facecolor='none',
    name='admin_1_states_provinces_lines'
)
ax.add_feature(states, edgecolor='gray')

# Add legend
plt.colorbar(img, ax=ax, shrink=0.7)